In [2]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback
from datetime import datetime
import os 
import traceback

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(parent_dir)
from robustness_prompts import get_prompt # noqa: E402

prompt_8_natural = get_prompt("natural", 8)

prompt_8_natural

'Simulate 1LYZ for 1ps at 300 K and for 1ps at 400K. Report the secondary structure assignments of the downloaded PDB structure and any known binding or active sites. Compute the RMSD, SASA, and radius of gyration of the 300K simulation.'

In [3]:
llm_model = "gpt-3.5-turbo-0125"
tools = "all"

In [4]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools, 
    use_memory=False,
    streaming=False,
    verbose=True,
)
with get_openai_callback() as cb:
    chat_start = datetime.now()
    try:
        response = agent.run(prompt_8_natural, callbacks=[cb])
        print(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = sys.exc_info()
        print(f'{type(e).__name__}:{e}')
        print("".join(traceback.format_exception(exc_type, exc_value, exc_traceback)))
    chat_end = datetime.now()
    total_runtime = (chat_end - chat_start).total_seconds()
    print(cb)
    print(f"Total runtime: {total_runtime:.2f}s")



> Entering new AgentExecutor chain...
Thought: To address the question effectively, I will need to perform multiple simulations, analyze the secondary structure assignments, identify any known binding or active sites, and compute the RMSD, SASA, and radius of gyration for the 300K simulation.

Action:
```
{
    "action": "SetUpandRunFunction",
    "action_input": {
        "pdb_id": "1LYZ",
        "forcefield_files": ["amber14/protein.ff14SB.xml", "amber14/tip3p.xml"],
        "save": true,
        "system_params": {
            "nonbondedMethod": "NoCutoff",
            "constraints": "HBonds",
            "rigidWater": true
        },
        "integrator_params": {
            "integrator_type": "LangevinMiddle",
            "Temperature": "300 * kelvin",
            "Friction": "1.0 / picoseconds",
            "Timestep": "0.002 * picoseconds",
            "Pressure": "1.0 * bar"
        },
        "simulation_params": {
            "Ensemble": "NVT",
            "Number of Steps

In [4]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions().replace(",", "\n"))

Files found in registry: 1LYZ_162451: PDB file downloaded from RSCB
 PDBFile ID: 1LYZ_162451
 1LYZ_162457: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 top_sim0_162459: Initial positions for simulation sim0_162459
 sim0_162459: Basic Simulation of Protein 1LYZ_162457
 rec0_162500: Simulation trajectory for protein 1LYZ_162457 and simulation sim0_162459
 rec1_162500: Simulation state log for protein 1LYZ_162457 and simulation sim0_162459
 rec2_162500: Simulation pdb frames for protein 1LYZ_162457 and simulation sim0_162459


In [5]:
# make sure pdb was downloaded
assert os.path.exists(registry.get_mapped_path("1LYZ_162451"))

In [9]:
# # make sure dssp was computed correctly
# from mdcrow.tools.base_tools import ComputeDSSP

# dssp = ComputeDSSP(registry)
# dssp._run(traj_file= "1LYZ_162451", target_frames="first")

In [8]:
# # make sure the sites were found
# from mdcrow.tools.base_tools import GetAllKnownSites

# get_all_known_sites = GetAllKnownSites()
# get_all_known_sites._run(query="1LYZ", primary_accession="")

In [10]:
# make sure trajectory and topology exist
traj_path_1 = registry.get_mapped_path("rec0_162500")
top_path_1 = registry.get_mapped_path("top_sim0_162459")

assert os.path.exists(traj_path_1)
assert os.path.exists(top_path_1)

In [11]:
# # make sure rmsd plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_022913'))

In [12]:
# # make sure rgy plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023033'))

In [13]:
# # make sure sasa plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023320'))

In [14]:
# # make sure trajectory and topology exist (sim2)
# traj_path_2 = registry.get_mapped_path("rec0_023742")
# top_path_2 = registry.get_mapped_path("top_sim0_023742")

# assert os.path.exists(traj_path_2)
# assert os.path.exists(top_path_2)

In [15]:
#verify the total cost
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

llm_cost = calculate_llm_cost(59624, 1251, "gpt-3.5-turbo-0125")
print('Input tokens:',59624)
print('Output tokens:',1251)
print('LLM costs: $',llm_cost)

Input tokens: 59624
Output tokens: 1251
LLM costs: $ 0.031689
